In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from tqdm import tqdm
import json

UA_LIST = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:118.0) Gecko/20100101 Firefox/118.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124 Safari/537.36",
]

def safe_request(url, max_retry=5):
    for _ in range(max_retry):
        try:
            headers = {"User-Agent": random.choice(UA_LIST)}
            res = requests.get(url, headers=headers, timeout=8)
            if res.status_code == 200:
                return res
        except:
            pass
        time.sleep(random.uniform(1, 2))
    print(f"[失败] 请求失败: {url}")
    return None


def extract_author_intro(soup):
    titles = soup.find_all("h2")
    for t in titles:
        if "作者简介" in t.text:
            intro = t.find_next("div", class_="intro")
            if intro:
                return intro.get_text(strip=True)

    intros = soup.select("#content .related_info .indent .intro")
    if len(intros) >= 2:
        return intros[1].get_text(strip=True)
    elif len(intros) == 1:
        text = intros[0].get_text(strip=True)
        if "出版" not in text and len(text) < 500:
            return text

    candidates = soup.select(".intro")
    for c in candidates:
        text = c.get_text(strip=True)
        if "作者" in text[:30]:  
            return text

    return ""


def fetch_hot_comments(book_link):
    comments_url = book_link.rstrip("/") + "/comments/"
    res = safe_request(comments_url)
    if not res:
        return "[]"

    soup = BeautifulSoup(res.text, "lxml")
    blocks = soup.select(".comment")[:5]

    comments_list = []
    for cb in blocks:
        text_tag = cb.select_one(".comment-content")
        vote_tag = cb.select_one(".vote-count")

        text = text_tag.get_text(strip=True) if text_tag else ""
        vote = vote_tag.get_text(strip=True) if vote_tag else ""

        if text:
            comments_list.append({"comment": text, "vote": vote})

    return json.dumps(comments_list, ensure_ascii=False)


df_base = pd.read_excel("豆瓣读书Top250.xlsx")

book_details = []

for idx, row in tqdm(df_base.iterrows(), total=len(df_base), desc="爬详情页"):

    title = row["书名"]
    link = row["链接"]
    author_info = row["作者/出版信息"]
    rating = row["评分"]
    short_quote = row["简介"]

    res = safe_request(link)
    if not res:
        book_details.append([title, link, author_info, rating, short_quote, "", "", "", ""])
        continue

    soup = BeautifulSoup(res.text, "lxml")

    intro_tag = soup.select_one("#link-report .intro")
    long_intro = intro_tag.get_text(strip=True) if intro_tag else ""

    author_intro = extract_author_intro(soup)

    .
    
    recommendations = []
    rec_section = soup.select_one("#db-rec-section")

    if rec_section:
        rec_items = rec_section.select("dl")[:10]
        for item in rec_items:
            name = item.select_one("dd a")
            score = item.select_one(".subject-rate")
            if name:
                recommendations.append({
                    "title": name.get_text(strip=True),
                    "rate": score.get_text(strip=True) if score else "",
                    "link": name["href"]
                })

    rec_json = json.dumps(recommendations, ensure_ascii=False)

    hot_comments = fetch_hot_comments(link)

    book_details.append([
        title, link, author_info, rating, short_quote,
        long_intro, author_intro, rec_json, hot_comments
    ])

    time.sleep(random.uniform(1, 2))

df_detail = pd.DataFrame(book_details, columns=[
    "书名", "链接", "作者/出版信息", "评分", "短简介",
    "长简介", "作者简介", "相关推荐", "短评"
])

df_detail.to_excel("豆瓣读书Top250_详情版.xlsx", index=False)

print("\n完成！已保存：豆瓣读书Top250_详情版.xlsx")


爬详情页: 100%|██████████████████████████████████████████████████████████████████████| 250/250 [38:22<00:00,  9.21s/it]



完成！已保存：豆瓣读书Top250_详情版.xlsx


In [3]:
df_detail.head(10)

,书名,链接,作者/出版信息,评分,短简介,长简介,作者简介,相关推荐,短评
0,红楼梦,https://book.douban.com/subject/1007305/,[清] 曹雪芹 著 / 人民文学出版社 / 1996-12 / 59.70元,9.7,都云作者痴，谁解其中味？,《红楼梦》是一部百科全书式的长篇小说。以宝黛爱情悲剧为主线，以四大家族的荣辱兴衰为背景，描绘...,"曹雪芹,（？-1763，一作1764）清小说家。名霑，字梦阮，号雪芹、芹圃、芹溪。为满洲正白...","[{""title"": ""简爱（英文全本）"", ""rate"": ""8.5"", ""link"": ...","[{""comment"": ""盛衰之理，本为天命。然而人心就是如此。眼见得他起高楼，于是便不忍..."
1,活着,https://book.douban.com/subject/4913064/,余华 / 作家出版社 / 2012-8 / 28.00元,9.4,生的苦难与伟大,《活着(新版)》讲述了农村人福贵悲惨的人生遭遇。福贵本是个阔少爷，可他嗜赌如命，终于赌光了家...,余华，1960年出生，1983年开始写作。至今已经出版长篇小说4部，中短篇小说集6部，随笔集...,"[{""title"": ""许三观卖血记"", ""rate"": ""9.2"", ""link"": ""h...","[{""comment"": ""读《活着》的过程我很平静，除了有庆献血而死的描述让我呼吸急促。在..."
2,哈利·波特,https://book.douban.com/subject/24531956/,J.K.罗琳 (J.K.Rowling) / 苏农 / 人民文学出版社 / 2008-12-...,9.7,从9¾站台开始的旅程,《哈利·波特(共7册)(精)》编著者J.K.罗琳。《哈利·波特(共7册)(精)》内容提要：2...,J. K. 罗琳（J. K. Rowling， 1965- ），英国女作家，自小喜欢写作，当...,"[{""title"": ""福尔摩斯探案全集（上中下）"", ""rate"": ""9.3"", ""li...","[{""comment"": ""自然记得从小学到高中的暑假，坐在空调房间的地板上，一边吃西瓜一边..."
3,1984,https://book.douban.com/subject/4820710/,[英] 乔治·奥威尔 / 刘绍铭 / 北京十月文艺出版社 / 2010-4-1 / 28.00,9.4,栗树荫下，我出卖你，你出卖我,★村上春树以《1Q84》向本书致敬★著名学者刘绍铭经典译本内地首次出版★62种文字风靡110...,乔治•奥威尔（1903-1950）， ...,"[{""title"": ""动物农场"", ""rate"": ""9.3"", ""link"": ""htt...","[{""comment"": ""这是我看过最吓人的一本书。不是可怕而是深深的恐惧。"", ""vot..."
4,三体全集: 地球往事三部曲,https://book.douban.com/subject/6518605/,刘慈欣 / 重庆出版社 / 2012-1 / 168.00元,9.5,地球往事三部曲,《地球往事·三体》文化大革命如火如荼进行的同时，军方探寻外星文明的绝秘计划“红岸工程”取得了...,刘慈欣，祖籍河南，长于山西，中国科普作家协会会员，山西省作家协会会员，高级工程师。自1999...,"[{""title"": ""球状闪电"", ""rate"": ""8.7"", ""link"": ""htt...","[{""comment"": ""大刘，你打开了一扇门，门外的世界一片黑暗，这黑暗中蕴藏的故事，永..."
5,百年孤独,https://book.douban.com/subject/6082808/,[哥伦比亚] 加西亚·马尔克斯 / 范晔 / 南海出版公司 / 2011-6 / 39.50元,9.3,魔幻现实主义文学代表作,《百年孤独》是魔幻现实主义文学的代表作，描写了布恩迪亚家族七代人的传奇故事，以及加勒比海沿岸...,加西亚•马尔克斯（Gabriel García Márquez）1927年出生于哥伦比亚马格...,"[{""title"": ""霍乱时期的爱情"", ""rate"": ""9.0"", ""link"": ""...","[{""comment"": ""每个人都是孤独地出生，在这世间恍惚几十年并不漫长的日子转眼就远去..."
6,飘,https://book.douban.com/subject/1068920/,[美国] 玛格丽特·米切尔 / 李美华 / 译林出版社 / 2000-9 / 40.00元,9.3,革命时期的爱情，随风而逝,小说中的故事发生在1861年美国南北战争前夕。生活在南方的少女郝思嘉从小深受南方文化传统的熏...,"米切尔（Margaret Mitchell, 1900-1949）美国女作家。出生于美国南部...","[{""title"": ""简爱（英文全本）"", ""rate"": ""8.5"", ""link"": ...","[{""comment"": ""有一天我梦醒了，窗外是氤氲不化的雾气，我走到窗边，原来是夜里下了..."
7,动物农场,https://book.douban.com/subject/2035179/,[英] 乔治·奥威尔 / 荣如德 / 上海译文出版社 / 2007-3 / 10.00元,9.3,太阳底下并无新事,《动物农场》是奥威尔最优秀的作品之一，是一则入木三分的反乌托的政治讽喻寓言。农场的一群动物成...,乔治·奥威尔（George Orwell），本名埃里克·亚瑟·布莱尔（Eric Arthur...,"[{""title"": ""1984"", ""rate"": ""9.4"", ""link"": ""htt...","[{""comment"": ""再也分不清猪的脸和人的脸"", ""vote"": ""3584""}, ..."
8,房思琪的初恋乐园,https://book.douban.com/subject/27614904/,林奕含 / 北京联合出版公司 / 2018-2 / 45.00元,9.2,向死而生的文学绝唱,令人心碎却无能为力的真实故事。向死而生的文学绝唱 打动万千读者的年度华语小说。李银河 戴锦华...,林奕含（1991——2017），台湾作家。出生于台南，曾居台北。梦想是一面写小说，一面像大江...,"[{""title"": ""离开的，留下的"", ""rate"": ""8.9"", ""link"": ""...","[{""comment"": ""25岁生日这天把房思琪交付印厂，简体版2018年1月25号上市。..."
9,三国演义（全二册）,https://book.douban.com/subject/1019568/,[明] 罗贯中 / 人民文学出版社 / 1998-05 / 39.50元,9.3,是非成败转头空,《三国演义》又名《三国志演义》、《三国志通俗演义》，是我国小说史上最著名最杰出的长篇章回体历...,罗贯中（约1330—约1400），汉族，名本，字贯中，号湖海散人。山西太原人，一说钱塘（现在...,"[{""title"": ""天龙八部"", ""rate"": ""9.2"", ""link"": ""htt...","[{""comment"": ""这书我看得特慢，只要出现一个地名我就想在谷歌地形图上给找出来，然..."


In [9]:
import pandas as pd
import json
import ast
import re

def safe_parse_json(x):
    if isinstance(x, list):
        return x
    if not isinstance(x, str) or x.strip() == "":
        return []

    try:
        return json.loads(x)
    except:
        try:
            return ast.literal_eval(x)
        except:
            return []


def clean_text(text):
    
    if not isinstance(text, str):
        return ""

    text = text.replace("\r", " ").replace("\n", " ")

    text = re.sub(r"\s+", " ", text)

    text = text.replace("(展开全部)", "").replace("（展开全部）", "")

    return text.strip()


def clean_author_intro(text):
    """去掉像书评一样的作者简介，保留真正简介"""

    text = clean_text(text)

    bad_keywords = ["读完", "看完", "小说", "角色", "故事", "情节"]
    if sum(k in text for k in bad_keywords) >= 3:
        return "" 

    if len(text) > 2000:
        return ""

    return text


def clean_short_comments(lst):
    """短评结构化 + 清洗文本 + vote 转 int"""

    lst = safe_parse_json(lst)
    results = []

    for item in lst:
        if "comment" not in item:
            continue

        comment = clean_text(item["comment"])
        vote_raw = item.get("vote", "0")

        try:
            vote = int(vote_raw)
        except:
            vote = 0

        if comment:
            results.append({"comment": comment, "vote": vote})

    return results


def clean_recommendations(lst):

    lst = safe_parse_json(lst)
    clean_list = []

    for item in lst:
        title = clean_text(item.get("title", ""))
        link = item.get("link", "")
        rate = item.get("rate", "")

        if title and link:
            clean_list.append({
                "title": title,
                "rate": rate,
                "link": link
            })

    return clean_list

def clean_douban_file(input_path, output_path):
    df = pd.read_excel(input_path)

    print("开始清洗数据...\n")

    df["短评"] = df["短评"].apply(clean_short_comments)
    df["作者简介"] = df["作者简介"].apply(clean_author_intro)
    df["长简介"] = df["长简介"].apply(clean_text)
    df["相关推荐"] = df["相关推荐"].apply(clean_recommendations)

    df = df.fillna("")

    df.to_excel(output_path, index=False)
    print(f"\n清洗完成！已保存到：{output_path}")


if __name__ == "__main__":
    clean_douban_file("豆瓣读书Top250.xlsx",
                      "豆瓣读书Top250.xlsx")


开始清洗数据...


清洗完成！已保存到：豆瓣读书Top250_详情版_清洗后.xlsx


In [11]:
df.head()

,书名,链接,作者/出版信息,评分,短简介,长简介,作者简介,相关推荐,短评,作者,出版社,出版日期,定价
0,红楼梦,https://book.douban.com/subject/1007305/,[清] 曹雪芹 著 / 人民文学出版社 / 1996-12 / 59.70元,9.7,都云作者痴，谁解其中味？,《红楼梦》是一部百科全书式的长篇小说。以宝黛爱情悲剧为主线，以四大家族的荣辱兴衰为背景，描绘...,"曹雪芹,（？-1763，一作1764）清小说家。名霑，字梦阮，号雪芹、芹圃、芹溪。为满洲正白...","[{'title': '简爱（英文全本）', 'rate': '8.5', 'link': ...",[{'comment': '盛衰之理，本为天命。然而人心就是如此。眼见得他起高楼，于是便不忍...,[清] 曹雪芹 著,人民文学出版社,1996-12,59.70元
1,活着,https://book.douban.com/subject/4913064/,余华 / 作家出版社 / 2012-8 / 28.00元,9.4,生的苦难与伟大,《活着(新版)》讲述了农村人福贵悲惨的人生遭遇。福贵本是个阔少爷，可他嗜赌如命，终于赌光了家...,余华，1960年出生，1983年开始写作。至今已经出版长篇小说4部，中短篇小说集6部，随笔集...,"[{'title': '许三观卖血记', 'rate': '9.2', 'link': 'h...",[{'comment': '读《活着》的过程我很平静，除了有庆献血而死的描述让我呼吸急促。在...,余华,作家出版社,2012-8,28.00元
2,哈利·波特,https://book.douban.com/subject/24531956/,J.K.罗琳 (J.K.Rowling) / 苏农 / 人民文学出版社 / 2008-12-...,9.7,从9¾站台开始的旅程,《哈利·波特(共7册)(精)》编著者J.K.罗琳。《哈利·波特(共7册)(精)》内容提要：2...,J. K. 罗琳（J. K. Rowling， 1965- ），英国女作家，自小喜欢写作，当...,"[{'title': '福尔摩斯探案全集（上中下）', 'rate': '9.3', 'li...",[{'comment': '自然记得从小学到高中的暑假，坐在空调房间的地板上，一边吃西瓜一边...,J.K.罗琳 (J.K.Rowling),苏农,人民文学出版社,2008-12-1
3,1984,https://book.douban.com/subject/4820710/,[英] 乔治·奥威尔 / 刘绍铭 / 北京十月文艺出版社 / 2010-4-1 / 28.00,9.4,栗树荫下，我出卖你，你出卖我,★村上春树以《1Q84》向本书致敬★著名学者刘绍铭经典译本内地首次出版★62种文字风靡110...,乔治•奥威尔（1903-1950）， ...,"[{'title': '动物农场', 'rate': '9.3', 'link': 'htt...","[{'comment': '这是我看过最吓人的一本书。不是可怕而是深深的恐惧。', 'vot...",[英] 乔治·奥威尔,刘绍铭,北京十月文艺出版社,2010-4-1
4,三体全集: 地球往事三部曲,https://book.douban.com/subject/6518605/,刘慈欣 / 重庆出版社 / 2012-1 / 168.00元,9.5,地球往事三部曲,《地球往事·三体》文化大革命如火如荼进行的同时，军方探寻外星文明的绝秘计划“红岸工程”取得了...,刘慈欣，祖籍河南，长于山西，中国科普作家协会会员，山西省作家协会会员，高级工程师。自1999...,"[{'title': '球状闪电', 'rate': '8.7', 'link': 'htt...",[{'comment': '大刘，你打开了一扇门，门外的世界一片黑暗，这黑暗中蕴藏的故事，永...,刘慈欣,重庆出版社,2012-1,168.00元
